# ML-09 — Validation and Research Claim Audit

This notebook conducts a rigorous audit of validation design, model leakage, and scientific claims for **Lane 2: Content Refresh Prioritization**.
I test memorization by comparing random vs grouped splits, prove my test harness detects leakage, and rewrite claims to strictly adhere to decision-support language.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Staleness correlates with organic search decline
- **Label Origin:** The decline label is measured as a drop in GSC search impressions across consecutive calendar months.
- **Methodology Question:** Does the validation design account for evergreen content? While mid-staleness articles (90-180 days) show elevated decline (61.1%), very old content (181+ days) actually exhibits a lower decline rate (47.1%) due to survivorship bias. A validation design must segment by content type and position tier to avoid penalizing stable evergreen assets.

### Finding 2: Striking-distance pages represent highest recovery ROI
- **Label Origin:** Content positioned in ranks 4 to 10 (page 1 striking distance).
- **Methodology Question:** Does a random split artificially inflate striking-distance precision? Because individual domains maintain distinct search authority profiles, evaluating striking-distance performance across unseen clients is necessary to verify whether the signal generalizes.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
df['avg_position_clean'] = df['avg_position'].replace(0, np.nan)

print(f"Loaded {len(df):,} rows across {df['client_id'].nunique()} clients for validation audit.")

Loaded 30,000 rows across 32 clients for validation audit.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I benchmark a **Random Split** (which permits pages from the same client to appear across train and test) against an **Honest Grouped Split** ( on ).

In [2]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

numeric_features = ['impressions_90d', 'clicks_90d', 'avg_position_clean', 'days_since_last_update', 'word_count']
categorical_features = ['position_tier', 'content_type', 'main_intent']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# 1. Random Split (Naive)
train_r, test_r = train_test_split(df, test_size=0.25, random_state=42)
rf_random = Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))])
rf_random.fit(train_r, train_r['is_declining'])
auc_random = roc_auc_score(test_r['is_declining'], rf_random.predict_proba(test_r)[:, 1])

# 2. Grouped Split (Honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_g_idx, test_g_idx = next(gss.split(df, groups=df['client_id']))
train_g, test_g = df.iloc[train_g_idx], df.iloc[test_g_idx]
rf_grouped = Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))])
rf_grouped.fit(train_g, train_g['is_declining'])
auc_grouped = roc_auc_score(test_g['is_declining'], rf_grouped.predict_proba(test_g)[:, 1])

print(f"Split Comparison (Before vs After):")
print(f"  Naive Random Split ROC-AUC:  {auc_random:.4f}")
print(f"  Honest Grouped Split ROC-AUC: {auc_grouped:.4f}")
print(f"  Generalization Gap:          {auc_random - auc_grouped:.4f}")

Split Comparison (Before vs After):
  Naive Random Split ROC-AUC:  0.7353
  Honest Grouped Split ROC-AUC: 0.5986
  Generalization Gap:          0.1367


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

To confirm the validation harness is sensitive to leakage, I deliberately inject the forward outcome metric  into the training features and observe the artificial score inflation.

In [3]:
# Intentionally add leaky feature
leaky_num = numeric_features + ['trend_pct']
prep_leaky = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), leaky_num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

rf_leaky = Pipeline([('prep', prep_leaky), ('clf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))])
rf_leaky.fit(train_g, train_g['is_declining'])
auc_leaky = roc_auc_score(test_g['is_declining'], rf_leaky.predict_proba(test_g)[:, 1])

print(f"Leakage Audit Results:")
print(f"  Honest Model ROC-AUC: {auc_grouped:.4f}")
print(f"  Leaky Model ROC-AUC:  {auc_leaky:.4f}")
print("CONFIRMED: The test harness collapses when leaky forward signals are introduced (AUC jumps to near 1.0). In my clean model, trend_pct is strictly excluded.")

Leakage Audit Results:
  Honest Model ROC-AUC: 0.5986
  Leaky Model ROC-AUC:  0.9999
CONFIRMED: The test harness collapses when leaky forward signals are introduced (AUC jumps to near 1.0). In my clean model, trend_pct is strictly excluded.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Bold Sentence (Unsafe Causal Phrasing):
> *"Our machine learning model predicts Google's algorithm to prove that updating stale striking-distance content will restore search rankings and drive a massive boost in clicks."*

### Honest Rewritten Sentence (Safe Decision-Support Phrasing):
> *"In this dataset across 32 clients, trailing 90-day search impressions and ranking position were observed to correlate directionally with subsequent 30-day traffic decline. When evaluated on unseen clients, the Random Forest model prioritized at-risk articles at a Precision@50 of 0.74 (compared to the 0.52 base rate), providing a useful decision-support queue for editorial refresh workflows."*

In [4]:
# Claim validation verification check
p50 = test_g['is_declining'].iloc[np.argsort(-rf_grouped.predict_proba(test_g)[:, 1])[:50]].mean()
print(f"Verified out-of-sample Precision@50: {p50:.2f} (Base Rate: {test_g['is_declining'].mean():.2f})")
print("Claim language accurately matches measured empirical receipts.")

Verified out-of-sample Precision@50: 0.74 (Base Rate: 0.52)
Claim language accurately matches measured empirical receipts.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under  — then submit your repo URL on the card. Done.